In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("preprocess.csv")
df.head()

,Partner,SeniorCitizen,InternetService,PaymentMethod,PaperlessBilling,MonthlyCharges,StreamingMovies,DeviceProtection,Contract,tenure,Dependents,Churn
0,False,False,No,Mailed check,False,20.40,False,False,Two year,11,False,False
1,True,False,DSL,Bank transfer (automatic),True,68.05,False,True,Two year,61,True,False
2,True,True,Fiber optic,Credit card (automatic),False,71.55,False,False,Month-to-month,34,False,False
3,True,False,No,Mailed check,False,24.05,False,False,Two year,70,True,False
4,False,False,No,Electronic check,True,20.00,False,False,Month-to-month,19,True,False


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3620 entries, 0 to 3619
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Partner           3620 non-null   bool   
 1   SeniorCitizen     3620 non-null   bool   
 2   InternetService   3620 non-null   str    
 3   PaymentMethod     3620 non-null   str    
 4   PaperlessBilling  3620 non-null   bool   
 5   MonthlyCharges    3620 non-null   float64
 6   StreamingMovies   3620 non-null   bool   
 7   DeviceProtection  3620 non-null   bool   
 8   Contract          3620 non-null   str    
 9   tenure            3620 non-null   int64  
 10  Dependents        3620 non-null   bool   
 11  Churn             3620 non-null   bool   
dtypes: bool(7), float64(1), int64(1), str(3)
memory usage: 298.3 KB


In [4]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split,cross_val_score
import optuna

from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import confusion_matrix, roc_auc_score, f1_score, recall_score, precision_score, accuracy_score


In [5]:
x = df.drop('Churn',axis=1)
y = df['Churn']

In [6]:
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,shuffle=True,stratify=y)

In [7]:
cols = df.select_dtypes(include=['str']).columns
print(cols)

Index(['InternetService', 'PaymentMethod', 'Contract'], dtype='str')


In [8]:
col = ['InternetService', 'PaymentMethod', 'Contract']
ct = ColumnTransformer(transformers=[('ohe',OneHotEncoder(),col)],remainder='passthrough')
x_train = ct.fit_transform(x_train)
x_test = ct.transform(x_test)

In [9]:
def objective(trial):
    par = {
        'C': trial.suggest_float('C', 0.01, 10),
        'penalty': trial.suggest_categorical('penalty', ['l1', 'l2']),
        'solver': 'liblinear'  # Required for l1 penalty
    }
    
    log = LogisticRegression(**par)
    
    scores = cross_val_score(log, x_train, y_train, cv=5, scoring='f1', n_jobs=-1)
    
    return scores.mean()

In [10]:
study = optuna.create_study(direction='maximize',sampler=optuna.samplers.TPESampler())
study.optimize(objective, n_trials=100,show_progress_bar=True)

[I 2026-05-13 07:28:55,355] A new study created in memory with name: no-name-6cf0738d-f5f0-4b5d-9e81-3390e9c0edee


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-05-13 07:29:04,441] Trial 0 finished with value: 0.7701346804286866 and parameters: {'C': 0.8530572405081723, 'penalty': 'l2'}. Best is trial 0 with value: 0.7701346804286866.
[I 2026-05-13 07:29:08,829] Trial 1 finished with value: 0.7703890046687482 and parameters: {'C': 6.273932704496339, 'penalty': 'l2'}. Best is trial 1 with value: 0.7703890046687482.
[I 2026-05-13 07:29:08,937] Trial 2 finished with value: 0.7703890046687482 and parameters: {'C': 8.91440496054659, 'penalty': 'l2'}. Best is trial 1 with value: 0.7703890046687482.
[I 2026-05-13 07:29:08,996] Trial 3 finished with value: 0.7703890046687482 and parameters: {'C': 4.617162477883507, 'penalty': 'l2'}. Best is trial 1 with value: 0.7703890046687482.
[I 2026-05-13 07:29:09,055] Trial 4 finished with value: 0.7703890046687482 and parameters: {'C': 8.721237115839791, 'penalty': 'l2'}. Best is trial 1 with value: 0.7703890046687482.
[I 2026-05-13 07:29:09,142] Trial 5 finished with value: 0.7701256475270284 and param

In [11]:
print(study.best_params)
print(study.best_value)

{'C': 2.888689420218171, 'penalty': 'l1'}
0.7705355581070188


In [12]:
def objective(trial):
    par={
        'learning_rate':trial.suggest_float('learning_rate',0.01,0.2,step=0.01),
        'max_depth':trial.suggest_int('max_depth',1,12),
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000,step=50),
        "lambda": trial.suggest_float("lambda", 1e-8, 1.0, log=True),
        "alpha": trial.suggest_float("alpha", 1e-8, 1.0, log=True),
        "subsample": trial.suggest_float("subsample", 0.2, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.2, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "random_state": 42
    }
    xgb = XGBClassifier(**par)
    score = cross_val_score(xgb,x_train, y_train,cv=5, scoring='f1', n_jobs=-1)
    return score.mean()

In [13]:
study_xgb = optuna.create_study(direction='maximize',sampler=optuna.samplers.TPESampler())
study_xgb.optimize(objective,n_trials=200,show_progress_bar=True)

[I 2026-05-13 07:29:17,401] A new study created in memory with name: no-name-f459effc-680d-4fbb-8019-6093e2a54c0a


  0%|          | 0/200 [00:00<?, ?it/s]

[I 2026-05-13 07:29:19,481] Trial 0 finished with value: 0.7367272491651079 and parameters: {'learning_rate': 0.11, 'max_depth': 4, 'n_estimators': 450, 'lambda': 0.00016518742051386973, 'alpha': 1.261284276206706e-08, 'subsample': 0.35900153043374367, 'colsample_bytree': 0.28815872707064527, 'min_child_weight': 1}. Best is trial 0 with value: 0.7367272491651079.
[I 2026-05-13 07:29:20,020] Trial 1 finished with value: 0.7762802264608538 and parameters: {'learning_rate': 0.16, 'max_depth': 1, 'n_estimators': 250, 'lambda': 6.065296924653184e-07, 'alpha': 3.573877178110609e-05, 'subsample': 0.5257140586515262, 'colsample_bytree': 0.8449294986896838, 'min_child_weight': 1}. Best is trial 1 with value: 0.7762802264608538.
[I 2026-05-13 07:29:20,326] Trial 2 finished with value: 0.7496664216985474 and parameters: {'learning_rate': 0.09999999999999999, 'max_depth': 10, 'n_estimators': 250, 'lambda': 9.056994786313385e-06, 'alpha': 2.278576755303835e-08, 'subsample': 0.5121066301712043, 'col

In [14]:
print(study_xgb.best_params)
print(study_xgb.best_value)

{'learning_rate': 0.09, 'max_depth': 2, 'n_estimators': 150, 'lambda': 0.00021894234382910125, 'alpha': 0.4228355528872177, 'subsample': 0.8169414721832754, 'colsample_bytree': 0.4218275217786136, 'min_child_weight': 9}
0.7861289659269881


In [15]:
def objective(trial):
    kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly'])
    par = {
        
        'C': trial.suggest_float('C', 0.1, 10.0, log=True), 
        
        'gamma': trial.suggest_categorical('gamma', ['scale', 'auto']),

    }
    
    svc = SVC(**par)
    score = cross_val_score(svc, x_train, y_train, cv=3, n_jobs=-1, scoring='f1')
    
    return score.mean()


In [16]:
study_svc = optuna.create_study(direction='maximize',sampler=optuna.samplers.TPESampler())
study_svc.optimize(objective,n_trials=60,show_progress_bar=True)

[I 2026-05-13 07:30:16,174] A new study created in memory with name: no-name-d8201de2-3dc0-42a9-9c30-883b501cdcd4


  0%|          | 0/60 [00:00<?, ?it/s]

[I 2026-05-13 07:30:16,566] Trial 0 finished with value: 0.696931062357181 and parameters: {'kernel': 'poly', 'C': 0.14000147481211533, 'gamma': 'auto'}. Best is trial 0 with value: 0.696931062357181.
[I 2026-05-13 07:30:16,847] Trial 1 finished with value: 0.7369805535854329 and parameters: {'kernel': 'poly', 'C': 0.3972455780169026, 'gamma': 'scale'}. Best is trial 1 with value: 0.7369805535854329.
[I 2026-05-13 07:30:17,201] Trial 2 finished with value: 0.7207450254626856 and parameters: {'kernel': 'rbf', 'C': 0.19678003696571006, 'gamma': 'auto'}. Best is trial 1 with value: 0.7369805535854329.
[I 2026-05-13 07:30:17,492] Trial 3 finished with value: 0.7501275386023326 and parameters: {'kernel': 'rbf', 'C': 6.130330838490016, 'gamma': 'scale'}. Best is trial 3 with value: 0.7501275386023326.
[I 2026-05-13 07:30:17,836] Trial 4 finished with value: 0.7346371789469547 and parameters: {'kernel': 'rbf', 'C': 5.901593391075126, 'gamma': 'auto'}. Best is trial 3 with value: 0.75012753860

In [17]:
print(study_svc.best_params)
print(study_svc.best_value)

{'kernel': 'linear', 'C': 9.87821935552725, 'gamma': 'scale'}
0.7570701564361406


In [18]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

def objective(trial):
    par = {
        'n_estimators': trial.suggest_int("n_estimators", 100, 1000, step=50),
        'max_depth': trial.suggest_int("max_depth", 3, 15),
        'max_features': trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        
        'min_samples_split': trial.suggest_int("min_samples_split", 2, 20),
        'min_samples_leaf': trial.suggest_int("min_samples_leaf", 1, 10),
        'bootstrap': trial.suggest_categorical("bootstrap", [True, False]),
        'random_state': 42,
        'n_jobs': -1,
    }
    
    rf = RandomForestClassifier(**par)
    score = cross_val_score(rf, x_train, y_train, cv=5, scoring='f1', n_jobs=1)
    
    return score.mean()


In [19]:
study_rf = optuna.create_study(direction='maximize',sampler=optuna.samplers.TPESampler())

[I 2026-05-13 07:30:34,700] A new study created in memory with name: no-name-942a8bde-0c05-48c6-9afe-a28286eab62d


In [20]:
study_rf.optimize(objective,n_trials=100,show_progress_bar=True)

  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-05-13 07:30:42,508] Trial 0 finished with value: 0.7717878375909858 and parameters: {'n_estimators': 750, 'max_depth': 8, 'max_features': 'sqrt', 'min_samples_split': 17, 'min_samples_leaf': 10, 'bootstrap': False}. Best is trial 0 with value: 0.7717878375909858.
[I 2026-05-13 07:30:44,759] Trial 1 finished with value: 0.7712467969976167 and parameters: {'n_estimators': 150, 'max_depth': 8, 'max_features': None, 'min_samples_split': 7, 'min_samples_leaf': 7, 'bootstrap': True}. Best is trial 0 with value: 0.7717878375909858.
[I 2026-05-13 07:30:53,544] Trial 2 finished with value: 0.7675623704835914 and parameters: {'n_estimators': 800, 'max_depth': 15, 'max_features': None, 'min_samples_split': 2, 'min_samples_leaf': 6, 'bootstrap': True}. Best is trial 0 with value: 0.7717878375909858.
[I 2026-05-13 07:30:56,526] Trial 3 finished with value: 0.7644859573128959 and parameters: {'n_estimators': 250, 'max_depth': 13, 'max_features': None, 'min_samples_split': 19, 'min_samples_le

In [21]:
print(study_rf.best_params)
print(study_rf.best_value)

{'n_estimators': 350, 'max_depth': 5, 'max_features': 'log2', 'min_samples_split': 2, 'min_samples_leaf': 2, 'bootstrap': False}
0.7762865881190997


In [22]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score

def objective(trial):
    par = {
        'n_neighbors': trial.suggest_int('n_neighbors', 3, 30),
        'weights': trial.suggest_categorical('weights', ['uniform', 'distance']),
        
        'metric': trial.suggest_categorical('metric', ['euclidean', 'manhattan']),
        'algorithm': trial.suggest_categorical('algorithm', ['auto', 'ball_tree', 'brute']),
        'leaf_size': trial.suggest_int('leaf_size', 10, 50)
    }

    knn = KNeighborsClassifier(**par)
    
    score = cross_val_score(knn, x_train, y_train, cv=5, scoring='f1', n_jobs=-1)
    return score.mean()


In [23]:
study_knn = optuna.create_study(direction='maximize',sampler=optuna.samplers.TPESampler())
study_knn.optimize(objective,n_trials=100,show_progress_bar=True)

[I 2026-05-13 07:38:37,178] A new study created in memory with name: no-name-25957231-95ea-406e-9a9a-4c8d568bb709


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-05-13 07:38:42,541] Trial 0 finished with value: 0.7498115109659462 and parameters: {'n_neighbors': 18, 'weights': 'distance', 'metric': 'euclidean', 'algorithm': 'auto', 'leaf_size': 39}. Best is trial 0 with value: 0.7498115109659462.
[I 2026-05-13 07:38:46,499] Trial 1 finished with value: 0.7661581910961719 and parameters: {'n_neighbors': 23, 'weights': 'distance', 'metric': 'manhattan', 'algorithm': 'ball_tree', 'leaf_size': 34}. Best is trial 1 with value: 0.7661581910961719.
[I 2026-05-13 07:38:46,582] Trial 2 finished with value: 0.7348370827282611 and parameters: {'n_neighbors': 7, 'weights': 'uniform', 'metric': 'euclidean', 'algorithm': 'ball_tree', 'leaf_size': 13}. Best is trial 1 with value: 0.7661581910961719.
[I 2026-05-13 07:38:46,664] Trial 3 finished with value: 0.7490457400575001 and parameters: {'n_neighbors': 12, 'weights': 'uniform', 'metric': 'manhattan', 'algorithm': 'brute', 'leaf_size': 44}. Best is trial 1 with value: 0.7661581910961719.
[I 2026-05-1

In [24]:
print(study_knn.best_params)
print(study_knn.best_value)

{'n_neighbors': 23, 'weights': 'uniform', 'metric': 'manhattan', 'algorithm': 'auto', 'leaf_size': 33}
0.7708241851351858


In [25]:

def train_and_test(par, model):
    model = model(**par)

    model.fit(x_train,y_train)
    
    y_pred = model.predict(x_test)
    
    print("Accuracy:", accuracy_score(y_test, y_pred)) 
    print("F1 Score:", f1_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("ROC AUC:", roc_auc_score(y_test, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    


In [31]:
study.best_params

{'C': 2.888689420218171, 'penalty': 'l1'}

In [33]:
model = LogisticRegression(C= 2.888689420218171, penalty= 'l1',solver= 'liblinear')
model.fit(x_train,y_train)
    
y_pred = model.predict(x_test)
    
print("Accuracy:", accuracy_score(y_test, y_pred)) 
print("F1 Score:", f1_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    

Accuracy: 0.761049723756906
F1 Score: 0.7633378932968536
Recall: 0.7707182320441989
Precision: 0.7560975609756098
ROC AUC: 0.7610497237569062
Confusion Matrix:
 [[272  90]
 [ 83 279]]


c:\Users\kumar\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\kumar\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


In [27]:
train_and_test(study_xgb.best_params,XGBClassifier)

Accuracy: 0.7734806629834254
F1 Score: 0.783068783068783
Recall: 0.8176795580110497
Precision: 0.751269035532995
ROC AUC: 0.7734806629834254
Confusion Matrix:
 [[264  98]
 [ 66 296]]


In [28]:
train_and_test(study_svc.best_params,SVC)

Accuracy: 0.7361878453038674
F1 Score: 0.7541827541827542
Recall: 0.8093922651933702
Precision: 0.7060240963855422
ROC AUC: 0.7361878453038674
Confusion Matrix:
 [[240 122]
 [ 69 293]]


In [29]:
train_and_test(study_rf.best_params,RandomForestClassifier)

Accuracy: 0.7679558011049724
F1 Score: 0.7771883289124668
Recall: 0.8093922651933702
Precision: 0.7474489795918368
ROC AUC: 0.7679558011049724
Confusion Matrix:
 [[263  99]
 [ 69 293]]


In [30]:
train_and_test(study_knn.best_params,KNeighborsClassifier)

Accuracy: 0.755524861878453
F1 Score: 0.7692307692307693
Recall: 0.8149171270718232
Precision: 0.7283950617283951
ROC AUC: 0.755524861878453
Confusion Matrix:
 [[252 110]
 [ 67 295]]


XGB best model